# 🦜 Classifier Birds — Pipeline Completo

**Flujo del notebook:**
1. Instalar dependencias
2. Montar Google Drive y configurar rutas
3. *(Opcional)* Recortar imágenes con YOLO → `dataset/`
4. Entrenar ResNet18 · ResNet50 · VGG16 · VGG19  *(float16, solo guarda best.pt)*
5. Ver resultados

---
**Estructura esperada en Google Drive:**
```
MyDrive/
  classifier_birds/
    data/
      Acropternis_orthonyx/   ← imágenes originales (si vas a usar YOLO)
      dataset/                ← imágenes ya recortadas 224×224  ← subir aquí
        Acropternis_orthonyx/
        Adelomyia_melanogenys/
        ...
    checkpoints/              ← se crea automáticamente
      resnet18/best.pt
      resnet50/best.pt
      vgg16/best.pt
      vgg19/best.pt
    logs/                     ← se crea automáticamente
```
> Si ya tienes el `dataset/` procesado, **salta la Sección 3** y ve directo a la Sección 4.


## 1 · Instalación de dependencias

In [1]:
!pip install ultralytics tqdm pillow -q
print('✓ Dependencias instaladas')

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 2 · Imports y configuración de rutas

In [2]:
import json
import logging
import os
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(name)s] %(message)s',
    datefmt='%H:%M:%S',
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


## 3 · Montar Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EDITAR ESTA CELDA si tu carpeta tiene otro nombre/ruta      ║
# ╚══════════════════════════════════════════════════════════════╝

DRIVE_ROOT    = Path('/content/drive/MyDrive/classifier_birds')
SOURCE_DIR    = DRIVE_ROOT / 'data'           # imágenes originales por especie
DATASET_DIR   = DRIVE_ROOT / 'data/dataset'  # imágenes recortadas 224×224
CHECKPOINT_DIR= DRIVE_ROOT / 'checkpoints'
LOG_DIR       = DRIVE_ROOT / 'logs'
YOLO_PATH     = DRIVE_ROOT / 'models/yolo/yolo26n.pt'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Rutas configuradas:')
for nombre, ruta in [('SOURCE', SOURCE_DIR), ('DATASET', DATASET_DIR),
                     ('CHECKPOINTS', CHECKPOINT_DIR), ('LOGS', LOG_DIR)]:
    existe = '✓' if ruta.exists() else '✗ (no existe aún)'
    print(f'  {nombre:12s} {ruta}  {existe}')

---
## 4 · (Opcional) Recorte con YOLO

> **Salta esta sección si ya subiste el `dataset/` procesado a Drive.**

Detecta pájaros en `data/<especie>/`, recorta el bounding box y guarda en `data/dataset/<especie>/` con tamaño 224×224.

Ajusta `--bird-class` si tu modelo YOLO usa un vocabulario distinto al de COCO-80 (donde *bird = 14*).

In [ ]:
from PIL import Image
from ultralytics import YOLO

IMG_SIZE     = (224, 224)
BIRD_CLASS   = 14      # clase 'bird' en COCO-80
CONF_THRESH  = 0.25
PADDING      = 5
FALLBACK     = True    # True → usar imagen completa si no hay detección

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def detect_and_crop(
    source_dir=SOURCE_DIR, dataset_dir=DATASET_DIR,
    yolo_path=YOLO_PATH, bird_class=BIRD_CLASS,
    conf_thresh=CONF_THRESH, fallback=FALLBACK, padding=PADDING,
):
    dataset_dir.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(yolo_path))

    species_dirs = sorted(
        d for d in source_dir.iterdir()
        if d.is_dir() and d.resolve() != dataset_dir.resolve()
    )

    stats = {'saved': 0, 'fallback': 0, 'skipped': 0}

    for sp_dir in tqdm(species_dirs, desc='Especies'):
        out_dir = dataset_dir / sp_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)

        imgs = [p for p in sorted(sp_dir.iterdir()) if p.suffix.lower() in IMG_EXTENSIONS]

        for img_path in tqdm(imgs, desc=sp_dir.name, leave=False):
            out_path = out_dir / (img_path.stem + '.jpg')
            if out_path.exists():
                continue
            try:
                img = Image.open(img_path).convert('RGB')
            except Exception:
                stats['skipped'] += 1
                continue

            results = model.predict(str(img_path), verbose=False, conf=conf_thresh)

            best_conf, best_box = 0.0, None
            for r in results:
                if r.boxes is None:
                    continue
                for box in r.boxes:
                    if int(box.cls[0]) == bird_class and float(box.conf[0]) > best_conf:
                        best_conf = float(box.conf[0])
                        best_box = box.xyxy[0].tolist()

            if best_box:
                x1, y1, x2, y2 = best_box
                x1 = max(0, int(x1) - padding)
                y1 = max(0, int(y1) - padding)
                x2 = min(img.width,  int(x2) + padding)
                y2 = min(img.height, int(y2) + padding)
                img.crop((x1, y1, x2, y2)).resize(IMG_SIZE, Image.LANCZOS).save(out_path, 'JPEG', quality=95)
                stats['saved'] += 1
            elif fallback:
                img.resize(IMG_SIZE, Image.LANCZOS).save(out_path, 'JPEG', quality=95)
                stats['fallback'] += 1
            else:
                stats['skipped'] += 1

    print(f"\nRecortes guardados : {stats['saved']}")
    print(f"Fallback (completa): {stats['fallback']}")
    print(f"Descartadas        : {stats['skipped']}")


# detect_and_crop()   # ← descomentar para ejecutar

---
## 5 · Explorar el dataset

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'No se encontró {DATASET_DIR}. Verifica la ruta o ejecuta la Sección 4.')

clases = sorted(d.name for d in DATASET_DIR.iterdir() if d.is_dir())
num_classes = len(clases)
total_imgs = sum(len(list(p.iterdir())) for p in DATASET_DIR.iterdir() if p.is_dir())

print(f'Clases   : {num_classes}')
print(f'Imágenes : {total_imgs}')
print(f'Promedio : {total_imgs // max(num_classes, 1)} imgs/clase')

# Mostrar muestra
sample_classes = random.sample(clases, min(6, num_classes))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, cls in zip(axes.flat, sample_classes):
    imgs_cls = list((DATASET_DIR / cls).iterdir())
    if imgs_cls:
        img = Image.open(random.choice(imgs_cls)).convert('RGB')
        ax.imshow(img)
    ax.set_title(cls.replace('_', ' '), fontsize=9)
    ax.axis('off')
plt.suptitle('Muestra del dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6 · Definición de modelos

In [ ]:
from torchvision.models import (
    vgg16, VGG16_Weights,
    vgg19, VGG19_Weights,
    resnet18, ResNet18_Weights,
    resnet50, ResNet50_Weights,
)

def get_vgg16(num_classes, pretrained=True):
    m = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    return m

def get_vgg19(num_classes, pretrained=True):
    m = vgg19(weights=VGG19_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    return m

def get_resnet18(num_classes, pretrained=True):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def get_resnet50(num_classes, pretrained=True):
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

MODEL_FACTORIES = {
    'vgg16':    get_vgg16,
    'vgg19':    get_vgg19,
    'resnet18': get_resnet18,
    'resnet50': get_resnet50,
}

print('Modelos disponibles:', list(MODEL_FACTORIES))

---
## 7 · Utilidades de entrenamiento

In [ ]:
# ── Batch size automático ──────────────────────────────────────────────────

def find_batch_size(model, device, num_classes, img_size=224, start=512, minimum=1):
    """Encuentra el mayor batch que cabe en VRAM (prueba start → start//2 → … → minimum)."""
    if device.type != 'cuda':
        print(f'  CPU detectado → batch size = {minimum}')
        return minimum

    criterion = nn.CrossEntropyLoss()
    batch = start
    while batch >= minimum:
        try:
            torch.cuda.empty_cache()
            x = torch.randn(batch, 3, img_size, img_size, device=device)
            y = torch.randint(0, num_classes, (batch,), device=device)
            model.train()
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                loss = criterion(model(x), y)
            loss.backward()
            model.zero_grad(set_to_none=True)
            del x, y, loss
            torch.cuda.empty_cache()
            print(f'  ✓ Batch size automático: {batch}')
            return batch
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if 'out of memory' not in str(e).lower() and not isinstance(e, torch.cuda.OutOfMemoryError):
                raise
            batch //= 2
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    return minimum


# ── Epoch train / val ──────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  train', leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                out = model(inputs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(inputs)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        b = inputs.size(0)
        total_loss += loss.item() * b
        correct += out.argmax(1).eq(labels).sum().item()
        total += b
        pbar.set_postfix(loss=f'{loss.item():.3f}', acc=f'{100*correct/total:.1f}%')
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = device.type == 'cuda'
    for inputs, labels in tqdm(loader, desc='  val  ', leave=False):
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        if use_amp:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                out = model(inputs)
                loss = criterion(out, labels)
        else:
            out = model(inputs)
            loss = criterion(out, labels)
        total_loss += loss.item() * inputs.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
        total += inputs.size(0)
    return total_loss / total, 100.0 * correct / total


# ── Checkpoints ────────────────────────────────────────────────────────────

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, history, class_to_idx):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'epoch': epoch,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict() if scheduler else None,
        'scaler_state':    scaler.state_dict() if scaler else None,
        'history':         history,
        'class_to_idx':    class_to_idx,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler, scaler, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    if scheduler and ckpt.get('scheduler_state'):
        scheduler.load_state_dict(ckpt['scheduler_state'])
    if scaler and ckpt.get('scaler_state'):
        scaler.load_state_dict(ckpt['scaler_state'])
    return ckpt['epoch'], ckpt.get('history', [])


print('✓ Utilidades cargadas')


---
## 8 · Configuración del entrenamiento

Ajusta los hiperparámetros y elige qué modelos entrenar.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  HIPERPARÁMETROS — editar a gusto                            ║
# ╚══════════════════════════════════════════════════════════════╝

MODELOS_A_ENTRENAR = ['resnet18', 'resnet50', 'vgg16', 'vgg19']  # ResNet primero
NUM_EPOCHS         = 30
LEARNING_RATE      = 1e-4
WEIGHT_DECAY       = 1e-4
VAL_SPLIT          = 0.2      # 20% validación
NUM_WORKERS        = 2        # en Colab 2 es suficiente
PRETRAINED         = True     # pesos ImageNet
RESUME             = False    # True → reanudar desde último checkpoint
SPLIT_SEED         = 42       # semilla fija para reproducir el split
BATCH_START        = 512      # batch máximo a probar

print('Configuración:')
print(f'  Modelos    : {MODELOS_A_ENTRENAR}')
print(f'  Épocas     : {NUM_EPOCHS}')
print(f'  LR         : {LEARNING_RATE}')
print(f'  Val split  : {VAL_SPLIT}')
print(f'  Pretrained : {PRETRAINED}')
print(f'  Resume     : {RESUME}')


---
## 9 · Entrenamiento

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset ───────────────────────────────────────────────────────────────
ds_train_full = datasets.ImageFolder(str(DATASET_DIR), transform=train_tf)
ds_val_full   = datasets.ImageFolder(str(DATASET_DIR), transform=val_tf)
num_classes   = len(ds_train_full.classes)
n             = len(ds_train_full)
val_size      = int(n * VAL_SPLIT)
train_size    = n - val_size

# Mismo split en cada ejecución / reanudación
rng   = torch.Generator().manual_seed(SPLIT_SEED)
perm  = torch.randperm(n, generator=rng).tolist()
train_idx, val_idx = perm[:train_size], perm[train_size:]

print(f'Clases          : {num_classes}')
print(f'Total imágenes  : {n}')
print(f'Train           : {train_size}   Val: {val_size}')

In [ ]:
import gc

all_histories = {}  # guarda el historial de cada modelo

for model_name in MODELOS_A_ENTRENAR:
    print(f'\n{"="*60}')
    print(f'  Entrenando: {model_name.upper()}')
    print(f'{"="*60}')

    ckpt_dir     = CHECKPOINT_DIR / model_name
    model_log_dir = LOG_DIR / model_name
    model_log_dir.mkdir(parents=True, exist_ok=True)

    # ── Modelo ────────────────────────────────────────────────────
    # Pesos en float32; autocast se encarga de usar float16 en el forward
    model = MODEL_FACTORIES[model_name](num_classes=num_classes, pretrained=PRETRAINED)
    model = model.to(device)

    # ── Batch size automático ─────────────────────────────────────
    print('Calculando batch size óptimo...')
    batch_size = find_batch_size(model, device, num_classes, start=BATCH_START)

    # ── DataLoaders ───────────────────────────────────────────────
    pin = device.type == 'cuda'
    train_loader = DataLoader(
        Subset(ds_train_full, train_idx),
        batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True,
    )
    val_loader = DataLoader(
        Subset(ds_val_full, val_idx),
        batch_size=min(batch_size * 2, 512), shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin,
    )

    # ── Optimizer / Scheduler / Criterion / Scaler ─────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    # ── Resume ─────────────────────────────────────────────────────
    start_epoch = 0
    history     = []
    if RESUME:
        best_ckpt = ckpt_dir / 'best.pt'
        if best_ckpt.exists():
            start_epoch, history = load_checkpoint(best_ckpt, model, optimizer, scheduler, scaler, device)
            start_epoch += 1
            print(f'Reanudando desde época {start_epoch}')
        else:
            print('No hay checkpoint previo, iniciando desde cero')

    best_val_acc = max((m['val_acc'] for m in history), default=0.0)

    # ── Bucle de entrenamiento ─────────────────────────────────────
    for epoch in range(start_epoch, NUM_EPOCHS):
        print(f'\nÉpoca {epoch+1}/{NUM_EPOCHS}  [{model_name}]')

        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        vl_loss, vl_acc = val_epoch(model, val_loader, criterion, device)
        scheduler.step()

        ep = {
            'epoch':      epoch,
            'train_loss': round(tr_loss, 5),
            'train_acc':  round(tr_acc,  3),
            'val_loss':   round(vl_loss, 5),
            'val_acc':    round(vl_acc,  3),
            'lr':         round(optimizer.param_groups[0]['lr'], 8),
        }
        history.append(ep)

        print(
            f'  train → loss={tr_loss:.4f}  acc={tr_acc:.1f}%  |'
            f'  val   → loss={vl_loss:.4f}  acc={vl_acc:.1f}%'
        )

        # Solo guardar si mejora → evita llenar Drive
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            save_checkpoint(
                ckpt_dir / 'best.pt',
                model, optimizer, scheduler, scaler,
                epoch, history, ds_train_full.class_to_idx,
            )
            print(f'  ★ Nuevo mejor val_acc: {vl_acc:.1f}% → best.pt guardado')

        # Historial JSON
        with open(model_log_dir / 'history.json', 'w') as f:
            json.dump(history, f, indent=2)

    all_histories[model_name] = history
    print(f'\n✓ {model_name} finalizado. Mejor val_acc: {best_val_acc:.1f}%')

    # Liberar memoria GPU antes del siguiente modelo
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

print('\n🎉 Entrenamiento completo')


---
## 10 · Resultados

In [ ]:
import matplotlib.pyplot as plt

# Cargar historial desde JSON si all_histories está vacío (re-ejecución)
if not all_histories:
    for model_name in MODELOS_A_ENTRENAR:
        h_path = LOG_DIR / model_name / 'history.json'
        if h_path.exists():
            with open(h_path) as f:
                all_histories[model_name] = json.load(f)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'vgg16': '#2196F3', 'vgg19': '#4CAF50', 'resnet18': '#FF9800', 'resnet50': '#E91E63'}

for model_name, history in all_histories.items():
    epochs   = [h['epoch'] + 1 for h in history]
    tr_loss  = [h['train_loss'] for h in history]
    vl_loss  = [h['val_loss']   for h in history]
    tr_acc   = [h['train_acc']  for h in history]
    vl_acc   = [h['val_acc']    for h in history]
    c = colors.get(model_name, 'gray')

    axes[0, 0].plot(epochs, tr_loss,  color=c, linestyle='-',  label=model_name)
    axes[0, 1].plot(epochs, vl_loss,  color=c, linestyle='-',  label=model_name)
    axes[1, 0].plot(epochs, tr_acc,   color=c, linestyle='-',  label=model_name)
    axes[1, 1].plot(epochs, vl_acc,   color=c, linestyle='-',  label=model_name)

titles = ['Train Loss', 'Val Loss', 'Train Accuracy (%)', 'Val Accuracy (%)']
for ax, title in zip(axes.flat, titles):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Época')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'resultados.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en Drive → resultados.png')

In [ ]:
# Tabla resumen del mejor val_acc por modelo
print(f'\n{"Modelo":<12} {"Best val_acc":>12} {"Best epoch":>10}')
print('-' * 38)
for model_name, history in all_histories.items():
    best = max(history, key=lambda h: h['val_acc'])
    print(f'{model_name:<12} {best["val_acc"]:>11.2f}%  época {best["epoch"]+1:>3}')